# Aviation Accidents Analysis

You are part of a consulting firm that is tasked to do an analysis of commercial and passenger jet airline safety. The client (an airline/airplane insurer) is interested in knowing what types of aircraft (makes/models) exhibit low rates of total destruction and low likelihood of fatal or serious passenger injuries in the event of an accident. They are also interested in any general variables/conditions that might be at play. Your analysis will be based off of aviation accident data accumulated from the years 1948-2023. 

Our client is only interested in airplane makes/models that are professional builds and could potentially still be active. Assume a max lifetime of 40 years for a make/model retirement and make sure to filter your data accordingly (i.e. from 1983 onwards). They would also like separate recommendations for small aircraft vs. larger passenger models. **In addition, make sure that claims that you make are statistically robust and that you have enough samples when making comparisons between groups.**


In this summative assessment you will demonstrate your ability to:
- **Use Pandas to load, inspect, and clean the dataset appropriately.**
- **Transform relevant columns to create measures that address the problem at hand.**
- conduct EDA: visualization and statistical measures to systematically understand the structure of the data
- recommend a set of airplanes and makes conforming to the client's request and identify at least *two* factors contributing to airplane safety. You must provide supporting evidence (visuals, summary statistics, tables) for each claim you make.

### Make relevant library imports

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from IPython.display import display

## Data Loading and Inspection

### Load in data from the relevant directory and inspect the dataframe.
- inspect NaNs, datatypes, and summary statistics

In [2]:
DATA_DIR = Path("data")

# Find AviationData.csv (case-insensitive; also finds data/aviationData/AviationData.csv)
candidates = [p for p in DATA_DIR.rglob("*.csv") if p.name.lower() == "aviationdata.csv"]
if not candidates:
    candidates = [p for p in DATA_DIR.rglob("*.csv") if "aviation" in p.name.lower()]

if not candidates:
    raise FileNotFoundError("Could not find AviationData.csv under data/")

data_path = candidates[0]
print("Loading:", data_path)

# The NTSB file is typically latin-1 / cp1252, not UTF-8
def load_csv_robust(path):
    for enc in ["utf-8", "latin-1", "cp1252", "iso-8859-1"]:
        try:
            df = pd.read_csv(path, low_memory=False, encoding=enc)
            print(f"Loaded with encoding: {enc}")
            return df
        except UnicodeDecodeError as e:
            print(f"Failed with {enc}: {e}")
    print("Falling back to latin-1 with replacement")
    return pd.read_csv(
        path, low_memory=False, encoding="latin-1",
        encoding_errors="replace", on_bad_lines="warn",
    )

df = load_csv_robust(data_path)

print("\nShape:", df.shape)
print("\nDtypes:")
print(df.dtypes)
print("\nMissing values (top 25):")
print(df.isna().sum().sort_values(ascending=False).head(25))
print("\nSummary statistics:")
display(df.describe(include="all").T)
df.head()

Loading: data\AviationData.csv
Failed with utf-8: 'utf-8' codec can't decode byte 0x96 in position 18: invalid start byte
Loaded with encoding: latin-1

Shape: (88889, 31)

Dtypes:
Event.Id                      str
Investigation.Type            str
Accident.Number               str
Event.Date                    str
Location                      str
Country                       str
Latitude                      str
Longitude                     str
Airport.Code                  str
Airport.Name                  str
Injury.Severity               str
Aircraft.damage               str
Aircraft.Category             str
Registration.Number           str
Make                          str
Model                         str
Amateur.Built                 str
Number.of.Engines         float64
Engine.Type                   str
FAR.Description               str
Schedule                      str
Purpose.of.flight             str
Air.carrier                   str
Total.Fatal.Injuries      float64
Tot

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Event.Id,88889,87951,20001214X45071,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Investigation.Type,88889,2,Accident,85015,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Accident.Number,88889,88863,ERA22LA103,2,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Event.Date,88889,14782,1982-05-16,25,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Location,88837,27758,"ANCHORAGE, AK",434,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Country,88663,219,United States,82248,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Latitude,34382,25589,332739N,19,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Longitude,34373,27154,0112457W,24,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Code,50132,10374,NONE,1488,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Airport.Name,52704,24870,Private,240,NaN,NaN,NaN,NaN,NaN,NaN,NaN


,Event.Id,Investigation.Type,Accident.Number,Event.Date,Location,Country,Latitude,Longitude,Airport.Code,Airport.Name,...,Purpose.of.flight,Air.carrier,Total.Fatal.Injuries,Total.Serious.Injuries,Total.Minor.Injuries,Total.Uninjured,Weather.Condition,Broad.phase.of.flight,Report.Status,Publication.Date
0,20001218X45444,Accident,SEA87LA080,1948-10-24,"MOOSE CREEK, ID",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,UNK,Cruise,Probable Cause,NaN
1,20001218X45447,Accident,LAX94LA336,1962-07-19,"BRIDGEPORT, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,4.0,0.0,0.0,0.0,UNK,Unknown,Probable Cause,19-09-1996
2,20061025X01555,Accident,NYC07LA005,1974-08-30,"Saltville, VA",United States,36.922223,-81.878056,NaN,NaN,...,Personal,NaN,3.0,NaN,NaN,NaN,IMC,Cruise,Probable Cause,26-02-2007
3,20001218X45448,Accident,LAX96LA321,1977-06-19,"EUREKA, CA",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,2.0,0.0,0.0,0.0,IMC,Cruise,Probable Cause,12-09-2000
4,20041105X01764,Accident,CHI79FA064,1979-08-02,"Canton, OH",United States,NaN,NaN,NaN,NaN,...,Personal,NaN,1.0,2.0,NaN,0.0,VMC,Approach,Probable Cause,16-04-1980


## Data Cleaning

### Filtering aircrafts and events

We want to filter the dataset to include aircraft that the client is interested in an analysis of:
- inspect relevant columns
- figure out any reasonable imputations
- filter the dataset

In [3]:
# Client only wants airplanes
df = df[
    df["Aircraft.Category"].astype("string").str.strip().str.lower().eq("airplane")
].copy()

# Client only wants professional builds
df = df[
    df["Amateur.Built"].astype("string").str.strip().str.lower().eq("no")
].copy()

# Parse dates
df["Event.Date"] = pd.to_datetime(df["Event.Date"], errors="coerce")

# Remove events older than 40 years → assignment says "from 1983 onwards"
cutoff_date = pd.Timestamp("1983-01-01")
df = df[df["Event.Date"] >= cutoff_date].copy()

print("Cutoff date:", cutoff_date)
print("Shape after aircraft/date filters:", df.shape)

Cutoff date: 1983-01-01 00:00:00
Shape after aircraft/date filters: (21447, 31)


### Cleaning and constructing Key Measurables

Injuries and robustness to destruction are a key interest point for the client. Clean and impute relevant columns and then create derived fields that best quantifies what the client wishes to track. **Use commenting or markdown to explain any cleaning assumptions as well as any derived columns you create.**

**Construct metric for fatal/serious injuries**

*Hint:* Estimate the total number of passengers on each flight. The likelihood of serious / fatal injury can be estimated as a fraction from this.

In [4]:
injury_cols = [
    "Total.Fatal.Injuries",
    "Total.Serious.Injuries",
    "Total.Minor.Injuries",
    "Total.Uninjured",
]

# Assumption: missing injury counts treated as 0 (documented in markdown)
for col in injury_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce").fillna(0).clip(lower=0)

# Estimated passengers on each flight
df["Total.Aboard"] = df[injury_cols].sum(axis=1)

# Derived injury metric
df["Fatal.Serious.Injuries"] = df["Total.Fatal.Injuries"] + df["Total.Serious.Injuries"]
df["Fatal.Serious.Ratio"] = np.where(
    df["Total.Aboard"] > 0,
    df["Fatal.Serious.Injuries"] / df["Total.Aboard"],
    np.nan,
)
df["Any.Fatal.Serious.Injury"] = df["Fatal.Serious.Injuries"] > 0

print(df[["Total.Aboard", "Fatal.Serious.Injuries", "Fatal.Serious.Ratio"]].describe())

       Total.Aboard  Fatal.Serious.Injuries  Fatal.Serious.Ratio
count  21447.000000            21447.000000         20543.000000
mean       8.592297                0.926563             0.283971
std       35.910084                6.816650             0.431704
min        0.000000                0.000000             0.000000
25%        1.000000                0.000000             0.000000
50%        2.000000                0.000000             0.000000
75%        2.000000                1.000000             0.800000
max      588.000000              295.000000             1.000000


**Aircraft.Damage**
- identify and execute any cleaning tasks
- construct a derived column tracking whether an aircraft was destroyed or not.

In [5]:
damage = df["Aircraft.damage"].astype("string").str.strip().str.title()
damage = damage.replace({
    "Nan": pd.NA, "None": pd.NA, "Unknown": pd.NA, "Unk": pd.NA, "": pd.NA,
})
df["Aircraft.damage"] = damage

df["Is.Destroyed"] = damage.str.lower().map({
    "destroyed": True,
    "substantial": False,
    "minor": False,
})

# Numeric version used later for groupby/mean
df["Is.Destroyed.num"] = df["Is.Destroyed"].astype("float")

print(df["Aircraft.damage"].value_counts(dropna=False))
print(df["Is.Destroyed"].value_counts(dropna=False))

Aircraft.damage
Substantial    16990
Destroyed       2316
<NA>            1324
Minor            817
Name: count, dtype: Int64
Is.Destroyed
False    17807
True      2316
NaN       1324
Name: count, dtype: int64


### Investigate the *Make* column
- Identify cleaning tasks here
- List cleaning tasks clearly in markdown
- Execute the cleaning tasks
- For your analysis, keep Makes with a reasonable number (you can put the threshold at 50 though lower could work as well)

In [6]:
# Cleaning tasks:
#   1. Strip whitespace and uppercase
#   2. Map placeholders (UNK/NONE/UNKNOWN/NaN/"") to missing
#   3. Standardise common aliases
#   4. Keep only Makes with >= 50 records
df["Make"] = (
    df["Make"].astype("string").str.strip().str.upper()
      .replace(["NAN", "NONE", "UNKNOWN", "", "UNK"], pd.NA)
)

df["Make"] = df["Make"].replace({
    "MCDONNELL-DOUGLAS": "MCDONNELL DOUGLAS",
    "MCDONNELL DOUGLAS CORP": "MCDONNELL DOUGLAS",
    "MCDONNELL DOUGLAS CORPORATION": "MCDONNELL DOUGLAS",
    "BEECH AIRCRAFT": "BEECH",
    "BEECH AIRCRAFT CORP": "BEECH",
    "BEECHCRAFT": "BEECH",
    "CESSNA AIRCRAFT": "CESSNA",
    "PIPER AIRCRAFT": "PIPER",
    "AIR TRACTOR INC": "AIR TRACTOR",
    "AIR TRACTOR, INC.": "AIR TRACTOR",
})

make_counts = df["Make"].value_counts()
keep_makes = make_counts[make_counts >= 50].index
df = df[df["Make"].isin(keep_makes)].copy()

print("Makes kept:", len(keep_makes))
print(make_counts.head(20))

Makes kept: 35
Make
CESSNA                7156
PIPER                 3996
BEECH                 1449
BOEING                1264
AIR TRACTOR            426
MOONEY                 363
AIRBUS                 243
CIRRUS DESIGN CORP     220
BELLANCA               219
MAULE                  215
AERONCA                200
CHAMPION               158
EMBRAER                153
GRUMMAN                147
LUSCOMBE               141
CIRRUS                 137
STINSON                129
MCDONNELL DOUGLAS      110
NORTH AMERICAN         106
DEHAVILLAND             95
Name: count, dtype: Int64


### Inspect Model column
- Get rid of any NaNs.
- Inspect the column and counts for each model/make. Are model labels unique to each make?
- If not, create a derived column that is a unique identifier for a given plane type.

In [7]:
df["Model"] = (
    df["Model"].astype("string").str.strip().str.upper()
      .replace(["NAN", "NONE", "UNKNOWN", "", "UNK"], pd.NA)
)
df = df.dropna(subset=["Model"]).copy()

# Model labels are NOT unique across makes (e.g. "600" appears in multiple makes),
# so build a unique identifier per plane type.
df["Plane.Type"] = df["Make"] + " " + df["Model"]

display(df.groupby(["Make", "Model"]).size().sort_values(ascending=False).head(30))

Make                Model    
CESSNA              172          770
BOEING              737          403
CESSNA              152          316
                    182          304
                    172S         276
PIPER               PA28         273
CESSNA              172N         249
                    180          213
                    172M         180
                    150          179
PIPER               PA-18-150    175
                    PA-28-140    169
BEECH               A36          165
CIRRUS DESIGN CORP  SR22         144
CESSNA              172P         143
                    140          116
                    172R         109
                    170B         107
PIPER               PA-28-180    105
                    PA-28-161    102
CESSNA              210           96
CIRRUS              SR22          96
MOONEY              M20J          94
PIPER               PA-28-181     92
CESSNA              A185F         90
AERONCA             7AC           89
PIPER   

### Cleaning other columns
- there are other columns containing data that might be related to the outcome of an accident. We list a few here:
- Engine.Type
- Weather.Condition
- Number.of.Engines
- Purpose.of.flight
- Broad.phase.of.flight

Inspect and identify potential cleaning tasks in each of the above columns. Execute those cleaning tasks. 

**Note**: You do not necessarily need to impute or drop NaNs here.

In [8]:
def clean_categorical(series, placeholders=("UNK", "UNKNOWN", "NONE", "NAN", "")):
    s = series.astype("string").str.strip().str.upper()
    return s.replace(list(placeholders), pd.NA)

for col in ["Engine.Type", "Weather.Condition", "Purpose.of.flight", "Broad.phase.of.flight"]:
    df[col] = clean_categorical(df[col])

# Number.of.Engines: below 1 doesn't make sense → NaN
df["Number.of.Engines"] = pd.to_numeric(df["Number.of.Engines"], errors="coerce")
df.loc[df["Number.of.Engines"] < 1, "Number.of.Engines"] = np.nan

for col in ["Engine.Type", "Weather.Condition", "Number.of.Engines",
            "Purpose.of.flight", "Broad.phase.of.flight"]:
    print(f"\n--- {col} ---")
    print(df[col].value_counts(dropna=False).head(15))


--- Engine.Type ---
Engine.Type
RECIPROCATING      12860
<NA>                3326
TURBO PROP           934
TURBO FAN            705
TURBO JET             71
GEARED TURBOFAN       12
TURBO SHAFT            9
Name: count, dtype: Int64

--- Weather.Condition ---
Weather.Condition
VMC     14317
<NA>     2693
IMC       907
Name: count, dtype: Int64

--- Number.of.Engines ---
Number.of.Engines
1.0    13248
2.0     2477
NaN     2098
4.0       67
3.0       27
Name: count, dtype: int64

--- Purpose.of.flight ---
Purpose.of.flight
PERSONAL                     9862
<NA>                         3363
INSTRUCTIONAL                2411
AERIAL APPLICATION            727
BUSINESS                      410
POSITIONING                   269
SKYDIVING                     157
AERIAL OBSERVATION            148
OTHER WORK USE                121
BANNER TOW                     86
FLIGHT TEST                    73
FERRY                          72
EXECUTIVE/CORPORATE            66
GLIDER TOW                    

### Column Removal
- inspect the dataframe and drop any columns that have too many NaNs

In [9]:
min_non_null = 20000
non_null_counts = df.notna().sum().sort_values(ascending=False)
print(non_null_counts)

cols_to_keep = non_null_counts[non_null_counts > min_non_null].index
print(f"\nColumns kept: {len(cols_to_keep)} / {len(df.columns)}")

df_clean = df[cols_to_keep].copy()
print("Final shape:", df_clean.shape)

Event.Id                    17917
Investigation.Type          17917
Accident.Number             17917
Event.Date                  17917
Amateur.Built               17917
Aircraft.Category           17917
Total.Aboard                17917
Fatal.Serious.Injuries      17917
Model                       17917
Make                        17917
Total.Uninjured             17917
Total.Minor.Injuries        17917
Total.Serious.Injuries      17917
Total.Fatal.Injuries        17917
Plane.Type                  17917
Any.Fatal.Serious.Injury    17917
Country                     17916
Location                    17913
Registration.Number         17753
FAR.Description             17569
Injury.Severity             17199
Fatal.Serious.Ratio         17139
Publication.Date            17129
Aircraft.damage             16771
Is.Destroyed                16771
Is.Destroyed.num            16771
Latitude                    16015
Longitude                   16012
Number.of.Engines           15819
Weather.Condit

### Save DataFrame to csv
- its generally useful to save data to file/server after its in a sufficiently cleaned or intermediate state
- the data can then be loaded directly in another notebook for further analysis
- this helps keep your notebooks and workflow readable, clean and modularized

In [10]:
out_path = DATA_DIR / "cleaned_aviation_data.csv"
df_clean.to_csv(out_path, index=False, encoding="utf-8")
print("Saved:", out_path)
print("Shape:", df_clean.shape)

Saved: data\cleaned_aviation_data.csv
Shape: (17917, 0)
